## Prerequisites - GPU environment check

**Run the cell below first.** It checks the NVIDIA tools (`nvidia-smi`, `nvcc`, `nsys`, `ncu`) and the Python GPU packages (`numpy`, `numba`, `cupy`) this course needs, and prints how to fix anything missing.

- **OK** - ready to use.
- **MISSING** - a required tool; fix it before running this module.
- **optional** - only affects specific bonus paths; the workbooks skip these gracefully.

In [ ]:
# Prerequisites: verify the GPU toolchain before running this notebook.
# This finds check_gpu.py at the repository root and runs it.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

# Module 11 — Final project **solution** (worked example)

> **Read this only after you have tried the 5-phase loop yourself.** The skill this
> module trains is the *reasoning*, not the final kernel. If you read the answer
> before predicting and measuring, you skip the exact thing being graded.

This notebook is the **solution** companion to the guided notebook
[../../Session_AI_Native_GPU.ipynb](../../Session_AI_Native_GPU.ipynb). It builds and
validates the worked answer [mva_infer_optimized.cu](mva_infer_optimized.cu) and walks
through the five phases that produced it. The full write-up is in
[README.md](README.md).

Run it on the **Linux GPU lab machine**, from the `11/final-project/solutions/` directory.

## 0. Environment check

Confirm you have a GPU and a working `nvcc`. If either cell fails, switch to the GPU environment.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!nvcc --version

## 1. Baseline — your speedup denominator

Build and run the correct-but-slow baseline first. Every speedup below is measured
against **its** throughput, and its built-in CPU reference is the correctness gate the
optimized version must also pass.

In [ ]:
!nvcc -O3 -arch=native ../mva_infer_baseline.cu -o mva_baseline

In [ ]:
!./mva_baseline 20000000

## Phases 1–3 — SPECIFY, GENERATE, PREDICT

**Phase 1 — SPECIFY (before generating anything):**
- **Inputs:** `num_events × 4` floats. The baseline stores them **event-major** (array-of-structs); with one thread per event, neighbouring threads read addresses 4 floats apart → **uncoalesced**.
- **Output:** `num_events` floats in `[0, 1]` (sigmoid range).
- **Launch config:** grid-stride loop, 256 threads/block → one configuration covers any event count and any GPU.
- **Memory model:** the model (mean, istd, weights, biases) is small, read-only, and read by *every* thread → put it in `__constant__` memory. Input goes to global memory in **struct-of-arrays** layout so reads coalesce.
- **Metric:** M events/sec for the full H2D + kernel + D2H round trip (same window as the baseline).

**Phase 2 — GENERATE (the AI's job):** ask for a constant-memory model, an SoA input layout, and a grid-stride kernel, *keeping the same math and the same PASS/FAIL gate*. Result: [mva_infer_optimized.cu](mva_infer_optimized.cu).

**Phase 3 — PREDICT (before running):**
- **Bound:** heavily **memory-bound** — each event does ~40 FLOPs but must load 4 input floats. Coalescing the loads is the change most likely to matter; constant memory removes repeated global loads of the weights.
- **Expected speedup:** most of the win is from coalescing + the constant cache; the end-to-end number is capped by the PCIe transfer of the input array, which both versions pay.
- **Correctness risks to watch:** the SoA index `input[i * num_events + idx]` must match how the host filled the array; the grid-stride loop must still cover every event when `grid` is clamped; fast-math intrinsics were **deliberately not** used (they can push the sigmoid past the `1e-5` tolerance).

## 2. Build the optimized variant

[mva_infer_optimized.cu](mva_infer_optimized.cu) applies: struct-of-arrays input
(coalesced loads), model parameters in `__constant__` memory, and a grid-stride loop.

In [ ]:
!nvcc -O3 -arch=native mva_infer_optimized.cu -o mva_infer_optimized

## Phase 4 — VERIFY (correctness gate)

The optimized kernel must reproduce the baseline's `RESULT: PASS`. **Deliberately stress
it**: run an event count that is **not** a multiple of the block size, plus a tiny
workload — that is where dropped bounds checks and grid-stride off-by-one bugs hide
(they corrupt the *tail*, which the gate now samples). All three must print `RESULT: PASS`.

In [ ]:
!./mva_infer_optimized 20000000     # round number
!./mva_infer_optimized 19999999     # NOT a multiple of the block size
!./mva_infer_optimized 1000         # tiny workload

## Phase 5 — DIAGNOSE (profile)

Profile the variant and compare against the Phase-3 prediction. Expect **high achieved
occupancy** (256-thread blocks, small register use) and **global-memory throughput much
closer to peak** than the baseline — that is the evidence the SoA change did what you
predicted. The new bottleneck is almost certainly the **host↔device transfer** of the
input array, not the kernel.

In [ ]:
# Nsight Systems timeline (if available). Look at H2D/D2H vs kernel time.
!nsys profile --stats=true -o mva_opt_profile ./mva_infer_optimized 20000000 || echo 'nsys not available — use the built-in timing and ncu below'

In [ ]:
# Nsight Compute kernel-level metrics: achieved occupancy and memory throughput.
!ncu --set basic --launch-count 1 ./mva_infer_optimized 1000000 || echo 'ncu not available — use nsys and the built-in timing'

## What changed, and the honest takeaway

**What changed vs the baseline, and why:**
1. **Struct-of-arrays (SoA)** input so neighbouring threads read adjacent addresses → coalesced global loads.
2. **Model in `__constant__` memory** — small, read-only, read by every thread → ideal for the constant cache.
3. **Grid-stride loop** — one launch config handles any event count and any GPU with good occupancy.
4. **Same CPU reference and PASS/FAIL gate** as the baseline — the speedup is only "real" if correctness still holds.

**What the AI could get wrong here (and the process caught):**
- Silently keeping the AoS layout while *claiming* it coalesced — caught by reading the index expression, not the prose.
- Swapping in `__expf`/`__fdividef` and breaking the `1e-5` tolerance — caught by the gate.
- A grid-stride loop that drops the last partial tile — caught by the `19999999` run.

**Next iterations for your workbook:** unified memory + prefetch (Module 02) and overlapping
transfer with compute using streams (Module 03), since the bottleneck is now the transfer.

The fast kernel is only a few lines different from the slow one. The **value** was in the
four decisions in Phase 1 and the three checks in Phase 4 — none of which the AI can be
trusted to make for you. That is the whole point of the module.